# schedule_raw_re_constituents
Incremental sync: `constituents` → `9b2cc153-7dc1-4e9c-b9e2-6f0ef315ba12`

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb


In [ ]:
ENDPOINT_NAME = "constituents"
DATASET_ID    = "9b2cc153-7dc1-4e9c-b9e2-6f0ef315ba12"  # raw_re_constituents
                              # (schedule_raw_re_comm_preferences reads it by that name)
MERGE_KEY     = "id"

# Exact column schema Domo expects — enforces types and column order before upsert.
# Add/remove columns here if the dataset schema changes in Domo.
DEST_SCHEMA = {
    "id":                              "object",
    "date_added":                      "object",
    "date_modified":                   "object",
    "gives_anonymously":               "Int64",
    "inactive":                        "Int64",
    "lookup_id":                       "object",
    "name":                            "object",
    "type":                            "object",
    "address_id":                      "object",
    "address_address_lines":           "object",
    "address_city":                    "object",
    "address_constituent_id":          "object",
    "address_country":                 "object",
    "address_do_not_mail":             "Int64",
    "address_formatted_address":       "object",
    "address_inactive":                "Int64",
    "address_postal_code":             "object",
    "address_preferred":               "Int64",
    "address_state":                   "object",
    "address_type":                    "object",
    "spouse_is_head_of_household":     "Int64",
    "deceased":                        "object",
    "first":                           "object",
    "fundraiser_status":               "object",
    "gender":                          "object",
    "last":                            "object",
    "address_start":                   "object",
    "email_id":                        "object",
    "email_address":                   "object",
    "email_constituent_id":            "object",
    "email_do_not_email":              "object",
    "email_inactive":                  "object",
    "email_primary":                   "object",
    "email_type":                      "object",
    "title":                           "object",
    "phone_id":                        "object",
    "phone_constituent_id":            "object",
    "phone_do_not_call":               "object",
    "phone_inactive":                  "object",
    "phone_number":                    "object",
    "phone_primary":                   "object",
    "phone_type":                      "object",
    "age":                             "float64",
    "birthdate_d":                     "float64",
    "birthdate_m":                     "float64",
    "birthdate_y":                     "float64",
    "title_2":                         "object",
    "spouse_id":                       "object",
    "spouse_first":                    "object",
    "spouse_last":                     "object",
    "marital_status":                  "object",
    "deceased_date_d":                 "float64",
    "deceased_date_m":                 "float64",
    "deceased_date_y":                 "float64",
    "constituent_assigned_fundraisers":"object",
    "online_presence_id":              "object",
    "online_presence_address":         "object",
    "online_presence_constituent_id":  "object",
    "online_presence_inactive":        "object",
    "online_presence_primary":         "object",
    "online_presence_type":            "object",
    "middle":                          "object",
    "suffix":                          "object",
    "preferred_name":                  "object",
    "address_end":                     "object",
    "former_name":                     "object",
    "suffix_2":                        "object",
    "address_county":                  "object",
    "address_suburb":                  "object",
    "pulled_at_utc":                   "object",
    "_endpoint":                       "object",
    "_incremental_field":              "object",
    "_days_back":                      "Int64",
}


In [ ]:
from pandas.api.types import is_bool_dtype, is_datetime64_any_dtype, is_float_dtype

def _coerce_to_dtype(s: pd.Series, dtype) -> pd.Series:
    """Safe coercion to destination dtype — handles booleans-as-strings,
    nullable ints, floats, and objects."""
    td = str(dtype)

    if is_datetime64_any_dtype(dtype) or td.startswith("datetime64"):
        return pd.to_datetime(s, errors="coerce")

    if td in {"Int64","Int32","Int16","Int8","UInt64","UInt32","UInt16","UInt8"}:
        if is_bool_dtype(s.dtype):
            s = s.map({True: 1, False: 0})
        if s.dtype == object:
            st = s.astype("string").str.strip().str.lower()
            s  = st.replace({"true": "1", "false": "0", "": pd.NA})
        return pd.to_numeric(s, errors="coerce").astype(td)

    if td.startswith("float") or is_float_dtype(dtype):
        return pd.to_numeric(s, errors="coerce").astype(td)

    if td == "boolean":
        if s.dtype == object:
            st = s.astype("string").str.strip().str.lower()
            s  = st.map({"true": True, "false": False, "1": True, "0": False})
        return s.astype("boolean")

    try:
        return s.astype(dtype)
    except Exception:
        return s.astype("object")


def conform_df_to_schema(df: pd.DataFrame, schema: dict,
                          fill_value=pd.NA, drop_extra: bool = True) -> pd.DataFrame:
    """Enforce exact column set, dtypes, and order before writing to Domo.
    - Adds any missing columns (filled with fill_value)
    - Casts every column to its declared dtype
    - Drops columns not in schema (if drop_extra=True)
    - Returns columns in schema order
    """
    out = df.copy(deep=True)
    for col, dtype in schema.items():
        if col not in out.columns:
            out[col] = pd.Series([fill_value] * len(out), dtype=dtype)
        else:
            out[col] = _coerce_to_dtype(out[col], dtype)
    if drop_extra:
        out = out.reindex(columns=list(schema.keys()))
    else:
        extras = [c for c in out.columns if c not in schema]
        out = out[list(schema.keys()) + extras]
    return out


In [ ]:
# ── Step 1: fetch using watermark ─────────────────────────────────────────────
state_df  = _read_pipeline_state()
since     = _get_since_date(ENDPOINT_NAME, state_df)

cfg       = ENDPOINT_LOOKUP[ENDPOINT_NAME]
cfg_run   = dict(cfg)
cfg_run["params_base"] = dict(cfg.get("params_base") or {})
cfg_run["params_base"][cfg["incremental_candidates"][0]] = since
ENDPOINT_LOOKUP[ENDPOINT_NAME] = cfg_run

try:
    token_mgr = TokenManager(interactive=False)
    df_raw    = fetch_incremental(token_mgr=token_mgr, endpoint_name=ENDPOINT_NAME,
                                  days_back=0, session=requests.Session())
finally:
    ENDPOINT_LOOKUP[ENDPOINT_NAME] = cfg

if df_raw.empty:
    print("⚠️  No data returned — nothing to write.")
    _write_pipeline_state(ENDPOINT_NAME, "success", 0, state_df)
else:
    # ── Step 2: conform to schema ──────────────────────────────────────────────
    df_ready = conform_df_to_schema(df_raw, DEST_SCHEMA)
    assert list(df_ready.columns) == list(DEST_SCHEMA.keys()), "Schema mismatch!"
    print(df_ready.dtypes)
    print(f"\nRows to upsert: {len(df_ready):,}")

    # ── Step 3: upsert to Domo ─────────────────────────────────────────────────
    try:
        domo.write_dataframe(
            domo_safe_cast(df_ready),
            dataset=DATASET_ID,
            update_method="upsert",
            update_key=MERGE_KEY,
        )
        print(f"✅ Upserted {len(df_ready):,} rows → {DATASET_ID}")
        _write_pipeline_state(ENDPOINT_NAME, "success", len(df_ready), state_df)
    except Exception as e:
        print(f"❌ Upsert failed: {e}")
        _write_pipeline_state(ENDPOINT_NAME, "failed", 0, state_df)
        raise
